# 07 — Export du modèle final (FD001)

**Objectif** : désigner le meilleur modèle d'après l'évaluation, l'exporter avec tout ce qu'il faut pour l'utiliser de façon autonome (scaler, liste des capteurs, configuration), puis vérifier que cet export fonctionne réellement à partir de données brutes jamais vues sous cette forme.

In [1]:
# --- Imports et configuration ---
from pathlib import Path
import json
import sys
import shutil

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

from src.data.loaders import load_subset, SENSOR_COLUMNS

SUBSET = "FD001"
WINDOW = 30
RUL_CLIP = 125

RESULTS_DIR = PROJECT_ROOT / "reports" / "results"
MODELS_DL_DIR = PROJECT_ROOT / "models" / "dl"
MODELS_ML_DIR = PROJECT_ROOT / "models" / "ml"
EXPORT_DIR = PROJECT_ROOT / "models" / "final"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Identification du modèle gagnant

On reprend le score NASA calculé dans `06_evaluation` pour désigner objectivement le meilleur modèle, plutôt que de le choisir à l'œil.

In [2]:
def score_nasa(y_reel, y_predit):
    d = y_predit - y_reel
    return np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)


MODELES = {"Random Forest": "random_forest", "XGBoost": "xgboost", "LSTM": "lstm"}

resultats = []
for nom, fichier in MODELES.items():
    df = pd.read_csv(RESULTS_DIR / f"predictions_{fichier}_{SUBSET}.csv")
    score_moyen = score_nasa(df["RUL_reelle"], df["RUL_predite"]).mean()
    resultats.append({"modele": nom, "fichier": fichier, "score_nasa_moyen": score_moyen})

tableau = pd.DataFrame(resultats).sort_values("score_nasa_moyen").reset_index(drop=True)
print(tableau)

modele_gagnant = tableau.iloc[0]
print()
print(f"Modele retenu : {modele_gagnant['modele']} (score NASA moyen = {modele_gagnant['score_nasa_moyen']:.2f})")

          modele        fichier  score_nasa_moyen
0           LSTM           lstm          2.894242
1  Random Forest  random_forest         14.464378
2        XGBoost        xgboost         16.111757

Modele retenu : LSTM (score NASA moyen = 2.89)


**Interprétation** : le LSTM est confirmé comme meilleur modèle (score NASA moyen 2,89), loin devant Random Forest (14,46) et XGBoost (16,11) — cohérent avec `06_evaluation`. C'est donc lui qui est exporté.

## 2. Reconstitution du scaler

Le modèle attend des capteurs normalisés. On recalcule ici le scaler exactement comme dans `02_preparation` — ajusté uniquement sur le train — pour pouvoir le sauvegarder en tant qu'objet réutilisable, indépendant du notebook qui l'a créé.

In [3]:
with open(PROJECT_ROOT / "config" / "fd001_capteurs_exclus.json", encoding="utf-8") as f:
    capteurs_exclus = json.load(f)["capteurs_exclus"]

capteurs_retenus = [c for c in SENSOR_COLUMNS if c not in capteurs_exclus]

df_train_brut, _, _ = load_subset(SUBSET)

scaler = MinMaxScaler()
scaler.fit(df_train_brut[capteurs_retenus])

print(f"Scaler ajuste sur {len(capteurs_retenus)} capteurs.")

Scaler ajuste sur 15 capteurs.


## 3. Export des artefacts

Trois fichiers sont sauvegardés ensemble : le modèle, le scaler, et un fichier de configuration qui décrit comment les utiliser (fenêtre attendue, capteurs, seuil d'écrêtage).

In [4]:
if modele_gagnant["fichier"] == "lstm":
    source_modele = MODELS_DL_DIR / f"lstm_{SUBSET}.keras"
    destination_modele = EXPORT_DIR / f"modele_final_{SUBSET}.keras"
else:
    source_modele = MODELS_ML_DIR / f"{modele_gagnant['fichier']}_{SUBSET}.joblib"
    destination_modele = EXPORT_DIR / f"modele_final_{SUBSET}.joblib"

shutil.copy(source_modele, destination_modele)
joblib.dump(scaler, EXPORT_DIR / f"scaler_{SUBSET}.joblib")

config_export = {
    "subset": SUBSET,
    "modele": modele_gagnant["modele"],
    "type_modele": modele_gagnant["fichier"],
    "fichier_modele": destination_modele.name,
    "fichier_scaler": f"scaler_{SUBSET}.joblib",
    "fenetre": WINDOW,
    "capteurs_retenus": capteurs_retenus,
    "rul_clip": RUL_CLIP,
    "score_nasa_moyen_validation": float(modele_gagnant["score_nasa_moyen"]),
}
with open(EXPORT_DIR / f"config_{SUBSET}.json", "w", encoding="utf-8") as f:
    json.dump(config_export, f, ensure_ascii=False, indent=2)

print("Artefacts exportes dans", EXPORT_DIR)
for f in sorted(EXPORT_DIR.glob(f"*{SUBSET}*")):
    print(" -", f.name)

Artefacts exportes dans C:\cmapss-prediction-rul\models\final
 - config_FD001.json
 - modele_final_FD001.keras
 - scaler_FD001.joblib


**Interprétation** : trois fichiers suffisent à emporter le modèle ailleurs — le modèle entraîné, le scaler ajusté sur le train, et un fichier de configuration qui documente tout ce dont on a besoin pour l'utiliser (capteurs, taille de fenêtre, seuil d'écrêtage).

## 4. Vérification : prédiction à partir de données brutes

On simule une utilisation réelle : on repart des données brutes du test (pas des fichiers déjà préparés par les notebooks précédents), on applique uniquement les artefacts exportés, et on compare la prédiction obtenue à la vraie RUL.

In [5]:
# Rechargement complet depuis zero, comme le ferait quelqu'un qui n'a que le dossier models/final/
with open(EXPORT_DIR / f"config_{SUBSET}.json", encoding="utf-8") as f:
    config_charge = json.load(f)

scaler_charge = joblib.load(EXPORT_DIR / f"scaler_{SUBSET}.joblib")

_, df_test_brut, df_rul_test = load_subset(SUBSET)

capteurs = config_charge["capteurs_retenus"]
fenetre = config_charge["fenetre"]

# Un seul moteur test, a titre d'exemple : le moteur 1
moteur_exemple = df_test_brut[df_test_brut["unit_number"] == 1].sort_values("time_in_cycles")
sequence_brute = moteur_exemple[capteurs].iloc[-fenetre:]
sequence_normalisee = scaler_charge.transform(sequence_brute)

if config_charge["type_modele"] == "lstm":
    modele_charge = keras.models.load_model(EXPORT_DIR / config_charge["fichier_modele"])
    entree = sequence_normalisee.reshape(1, fenetre, len(capteurs))
    prediction = float(modele_charge.predict(entree, verbose=0).flatten()[0])
else:
    modele_charge = joblib.load(EXPORT_DIR / config_charge["fichier_modele"])
    stats = {}
    for i, capteur in enumerate(capteurs):
        col = sequence_normalisee[:, i]
        stats[f"{capteur}_moyenne"] = col.mean()
        stats[f"{capteur}_ecart_type"] = col.std()
        stats[f"{capteur}_min"] = col.min()
        stats[f"{capteur}_max"] = col.max()
    entree = pd.DataFrame([stats])
    prediction = float(modele_charge.predict(entree)[0])

vraie_rul = min(df_rul_test.loc[0, "RUL"], RUL_CLIP)

print(f"RUL predite (export autonome) : {prediction:.1f}")
print(f"RUL reelle (ecretee a {RUL_CLIP})   : {vraie_rul}")

RUL predite (export autonome) : 113.7
RUL reelle (ecretee a 125)   : 112


**Interprétation** : en repartant uniquement des données brutes du moteur 1 et des trois fichiers exportés, la prédiction obtenue (113,7) est très proche de la vraie RUL écrêtée (112) — l'export fonctionne de façon autonome, sans dépendre du notebook d'origine.

## Synthèse

Le modèle gagnant, son scaler et sa configuration sont sauvegardés dans `models/final/` — trois fichiers suffisants pour prédire une RUL à partir de mesures capteurs brutes, sans dépendre d'aucun notebook. La vérification en section 4 confirme que l'export fonctionne de façon autonome.